In [1]:
# ===============================
# IMPORT LIBRARIES
# ===============================
import pandas as pd
import numpy as np
import pickle

from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity



In [2]:

# ===============================
# LOAD DATASET
# ===============================
df = pd.read_csv('..\data\zomato.csv')



In [3]:

# ===============================
# SELECT REQUIRED COLUMNS
# ===============================
df = df[['name', 'cuisines', 'rate', 'approx_cost(for two people)', 'rest_type']]



In [4]:

# ===============================
# RENAME COLUMNS
# ===============================
df.rename(columns={
    'rate': 'Mean Rating',
    'approx_cost(for two people)': 'cost'
}, inplace=True)



In [5]:

# ===============================
# DATA CLEANING
# ===============================

# Fill missing values
df['cuisines'] = df['cuisines'].fillna('')
df['rest_type'] = df['rest_type'].fillna('')
df['Mean Rating'] = df['Mean Rating'].fillna(0)

# Clean cost (remove commas and convert to float)
df['cost'] = df['cost'].astype(str).str.replace(',', '')
df['cost'] = pd.to_numeric(df['cost'], errors='coerce').fillna(0)

# Clean rating (remove '/5' if present)
df['Mean Rating'] = df['Mean Rating'].astype(str).str.replace('/5', '')
df['Mean Rating'] = pd.to_numeric(df['Mean Rating'], errors='coerce').fillna(0)



In [6]:

# ===============================
# FEATURE ENGINEERING (IMPORTANT)
# ===============================

# Create tags for ML model
df['tags'] = df['cuisines'] + " " + df['rest_type']

# Lowercase
df['tags'] = df['tags'].str.lower()



In [7]:

# ===============================
# FINAL DATAFRAME (VERY IMPORTANT)
# ===============================
restaurants = df[['name', 'cuisines', 'Mean Rating', 'cost', 'tags']]



In [8]:

# ===============================
# VECTORIZATION
# ===============================
cv = CountVectorizer(max_features=5000, stop_words='english')
vectors = cv.fit_transform(restaurants['tags']).toarray()



In [9]:

# ===============================
# SIMILARITY
# ===============================
similarity = cosine_similarity(vectors)



In [10]:

# ===============================
# SAVE MODEL FILES
# ===============================
pickle.dump(restaurants, open('../models/restaurants.pkl', 'wb'))
pickle.dump(similarity, open('../models/similarity.pkl', 'wb'))



In [12]:

# ===============================
# FINAL CHECK (IMPORTANT)
# ===============================
print("Columns in model:")
print(restaurants.columns)

print("\nSample data:")
print(restaurants.head())

Columns in model:
Index(['name', 'cuisines', 'Mean Rating', 'cost', 'tags'], dtype='object')

Sample data:
                    name                        cuisines  Mean Rating   cost  \
0                  Jalsa  North Indian, Mughlai, Chinese          4.1  800.0   
1         Spice Elephant     Chinese, North Indian, Thai          4.1  800.0   
2        San Churro Cafe          Cafe, Mexican, Italian          3.8  800.0   
3  Addhuri Udupi Bhojana      South Indian, North Indian          3.7  300.0   
4          Grand Village        North Indian, Rajasthani          3.8  600.0   

                                           tags  
0  north indian, mughlai, chinese casual dining  
1     chinese, north indian, thai casual dining  
2    cafe, mexican, italian cafe, casual dining  
3        south indian, north indian quick bites  
4        north indian, rajasthani casual dining  
